In [13]:
import pandas as pd
df_merged=pd.read_csv("top10_segments_full_info.csv")

In [14]:
import pandas as pd
import folium
from folium.plugins import TimestampedGeoJson

# Giả định df_merged là DataFrame tổng hợp thu được từ bước merge ở cell trước của bạn.
# Để bản đồ demo chạy nhanh và không bị quá tải, ta sẽ lấy dữ liệu của 1 ngày làm mẫu (ví dụ 96 mốc)
# Bạn có thể bỏ dòng lọc này nếu muốn chạy toàn bộ 268 ngày
df_merged['updated_at'] = pd.to_datetime(df_merged['updated_at'])
df_sample = df_merged[df_merged['updated_at'] <= df_merged['updated_at'].min() + pd.Timedelta(days=1)].copy()

# Định dạng lại cột thời gian sang chuỗi chuẩn ISO để plugin Folium đọc được
df_sample['time_str'] = df_sample['updated_at'].dt.strftime('%Y-%m-%dT%H:%M:%S')

# 1. Hàm xác định màu sắc dựa trên tỷ lệ vận tốc so với max_velocity
def get_traffic_color(row):
    vel = row['velocity']
    max_vel = row['max_velocity']
    
    if pd.isna(vel) or pd.isna(max_vel) or max_vel == 0:
        return '#808080' # Màu xám nếu thiếu dữ liệu
        
    ratio = vel / max_vel
    if ratio < 0.40:
        return '#FF0000' # Đỏ (Kẹt xe nặng)
    elif ratio < 0.75:
        return '#FFD700' # Vàng (Chậm)
    else:
        return '#00FF00' # Xanh lá (Thông thoáng)

# Áp dụng hàm để tạo cột màu sắc trực tiếp trên tập dữ liệu
df_sample['color'] = df_sample.apply(get_traffic_color, axis=1)

# 2. Khởi tạo bản đồ Folium tại tọa độ trung tâm của dữ liệu
center_lat = df_sample['start_node_lat'].mean()
center_long = df_sample['start_node_long'].mean()
m = folium.Map(location=[center_lat, center_long], zoom_start=14, tiles='CartoDB positron')

# 3. Tạo cấu trúc các Feature cho GeoJSON động
features = []

for _, row in df_sample.iterrows():
    # Tạo cặp tọa độ đường thẳng nối từ Node bắt đầu đến Node kết thúc
    # Lưu ý: GeoJSON yêu cầu [Kinh độ (Long), Vĩ độ (Lat)]
    coordinates = [
        [row['start_node_long'], row['start_node_lat']],
        [row['end_node_long'], row['end_node_lat']]
    ]
    
    # Đóng gói thông tin của từng con đường tại mốc thời gian t
    feature = {
        'type': 'Feature',
        'geometry': {
            'type': 'LineString',
            'coordinates': coordinates
        },
        'properties': {
            'times': [row['time_str'], row['time_str']], # Thời gian bắt đầu và kết thúc hiển thị của mốc này
            'style': {
                'color': row['color'],
                'weight': 6,          # Độ dày của con đường trên map
                'opacity': 0.8
            },
            'popup': f"Đường: {row['street_name']} | Vận tốc: {round(row['velocity'], 1)} km/h"
        }
    }
    features.append(feature)

geojson_data = {
    'type': 'FeatureCollection',
    'features': features
}

# 4. Tích hợp Plugin Tua thời gian (TimestampedGeoJson) vào bản đồ
TimestampedGeoJson(
    geojson_data,
    period='PT15M',          # Chu kỳ bước nhảy thời gian là 15 phút (Period Time 15 Minutes)
    duration='PT15M',        # Thời gian tồn tại của vạch màu trước khi sang mốc mới
    add_last_point=False,
    auto_play=True,          # Tự động chạy khi mở bản đồ
    loop=True,               # Lặp lại khi chạy hết chuỗi ngày
    max_speed=1,             # Tốc độ phát: cứ 1 giây cập nhật sang mốc tiếp theo đúng yêu cầu của bạn
    time_slider_drag_update=True
).add_to(m)

# 5. Lưu bản đồ ra file HTML duy nhất
output_map_name = 'traffic_sliding_window_map.html'
m.save(output_map_name)

print(f"Đã tạo xong bản đồ giao thông động 10 tuyến đường!")
print(f"Hãy mở file '{output_map_name}' bằng trình duyệt của bạn và nhấn nút Play để xem map cập nhật màu sắc mỗi giây.")

Đã tạo xong bản đồ giao thông động 10 tuyến đường!
Hãy mở file 'traffic_sliding_window_map.html' bằng trình duyệt của bạn và nhấn nút Play để xem map cập nhật màu sắc mỗi giây.
